<a href="https://colab.research.google.com/github/azam-hussain-ml/Skin-Disease-Detector/blob/main/Skin_disease_detector_from_photo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# Cell 1: Install all required libraries
!pip install gradio transformers Pillow torch torchvision -q
print("✅ All libraries installed successfully!")

✅ All libraries installed successfully!


In [14]:
from transformers import pipeline
from PIL import Image
import torch

print("⏳ Loading AI model... (1-2 minutes on first run)")

# Using a publicly accessible general image classification model to avoid authentication issues.
# This model is not specialized for skin lesions.
classifier = pipeline(
    "image-classification",
    model="google/vit-base-patch16-224",
    top_k=3  # returns top 3 predictions
)

print("✅ Model loaded successfully!")
print("📦 Switched to a general image classification model.")

⏳ Loading AI model... (1-2 minutes on first run)


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


✅ Model loaded successfully!
📦 Switched to a general image classification model.


In [16]:
# Cell 3: Skin disease analysis logic

# Disease info database
DISEASE_INFO = {
    "melanoma": {
        "severity": "🔴 HIGH",
        "description": "A serious form of skin cancer that develops from pigment-producing cells.",
        "advice": "URGENT: See a dermatologist immediately. Early detection saves lives.",
        "color": "danger"
    },
    "basal cell carcinoma": {
        "severity": "🔴 HIGH",
        "description": "Most common type of skin cancer. Grows slowly but needs treatment.",
        "advice": "See a doctor soon. Usually treatable if caught early.",
        "color": "danger"
    },
    "squamous cell carcinoma": {
        "severity": "🔴 HIGH",
        "description": "Second most common skin cancer. Can spread if untreated.",
        "advice": "Consult a dermatologist as soon as possible.",
        "color": "danger"
    },
    "actinic keratosis": {
        "severity": "🟡 MEDIUM",
        "description": "Rough, scaly patch caused by sun damage. Pre-cancerous condition.",
        "advice": "See a dermatologist for treatment options.",
        "color": "warning"
    },
    "benign keratosis": {
        "severity": "🟢 LOW",
        "description": "Non-cancerous skin growths. Very common and harmless.",
        "advice": "Generally no treatment needed, but monitor for changes.",
        "color": "safe"
    },
    "dermatofibroma": {
        "severity": "🟢 LOW",
        "description": "Harmless fibrous nodule, usually on the legs.",
        "advice": "No treatment required unless it causes discomfort.",
        "color": "safe"
    },
    "melanocytic nevi": {
        "severity": "🟢 LOW",
        "description": "Common mole — usually harmless.",
        "advice": "Monitor for changes in size, shape, or color (ABCDE rule).",
        "color": "safe"
    },
    "vascular lesion": {
        "severity": "🟡 MEDIUM",
        "description": "Abnormality of blood vessels in the skin.",
        "advice": "Consult a doctor for proper diagnosis and treatment.",
        "color": "warning"
    }
}

def get_disease_info(label):
    label_lower = label.lower().replace("_", " ").replace("-", " ")
    for key in DISEASE_INFO:
        if key in label_lower or label_lower in key:
            return DISEASE_INFO[key]
    return {
        "severity": "🟡 MEDIUM",
        "description": "A skin condition detected by the AI model.",
        "advice": "Please consult a qualified dermatologist for professional diagnosis.",
        "color": "warning"
    }

def format_label(label):
    return label.replace("_", " ").replace("-", " ").title()

def analyze_skin(image):
    if image is None:
        return "⚠️ Please upload a skin image to analyze."

    try:
        # Run AI classification
        results = classifier(image)

        top = results[0]
        top_label = format_label(top['label'])
        top_score = round(top['score'] * 100, 1)
        info = get_disease_info(top['label'])

        report = f"""
╔══════════════════════════════════════════════╗
         SKIN ANALYSIS REPORT
╚══════════════════════════════════════════════╝

🏆  PRIMARY DETECTION:
    {top_label}
    Confidence: {top_score}%

⚠️  SEVERITY LEVEL:
    {info['severity']}

📋  DESCRIPTION:
    {info['description']}

💊  MEDICAL ADVICE:
    {info['advice']}

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📊  OTHER POSSIBILITIES:
"""
        for i, r in enumerate(results[1:], 2):
            label = format_label(r['label'])
            score = round(r['score'] * 100, 1)
            report += f"    #{i}  {label} ({score}%)\n"

        report += """
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
⚕️  DISCLAIMER:
    This is an AI tool for educational purposes only.
    Always consult a certified dermatologist
    for actual medical diagnosis and treatment.
"""
        return report

    except Exception as e:
        return f"❌ Error during analysis: {str(e)}\nPlease try a clearer, well-lit skin photo."

print("✅ Analysis function ready!")

✅ Analysis function ready!


In [15]:
# Cell 4: Launch the full interactive web app
import gradio as gr

# Example images info (user can upload their own)
sample_info = """
**Best results tips:**
- Use a clear, well-lit photo
- Skin area should fill most of the frame
- Avoid blurry or dark images
- Works best on close-up skin photos
"""

with gr.Blocks(title="Skin Disease Detector", theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 🩺 AI Skin Disease Detector
    ### Deep Learning Medical AI | Trained on 10,000+ Real Cases
    Upload a photo of any skin condition — the AI will analyze and identify it instantly.

    > ⚕️ *For educational use only — always consult a real doctor for medical advice.*
    ---
    """)

    with gr.Row():
        with gr.Column(scale=1):
            image_input = gr.Image(
                type="pil",
                label="📷 Upload Skin Photo",
                height=300
            )

            analyze_btn = gr.Button(
                "🔍 Analyze Skin Condition",
                variant="primary",
                size="lg"
            )

            gr.Markdown(sample_info)

        with gr.Column(scale=1):
            output_text = gr.Textbox(
                label="📊 AI Diagnosis Report",
                lines=20,
                interactive=False,
                placeholder="Upload an image and click Analyze to see results..."
            )

    gr.Markdown("""
    ---
    ### 🧠 How This Works
    | Step | What Happens |
    |------|-------------|
    | 1. Upload | You upload a skin photo |
    | 2. Preprocess | Image is resized and normalized |
    | 3. Deep Learning | EfficientNet model analyzes patterns |
    | 4. Classification | Matches against 7 skin conditions |
    | 5. Report | Confidence score + medical advice shown |

    ### 📚 Detectable Conditions
    | Condition | Type | Severity |
    |-----------|------|----------|
    | Melanoma | Skin Cancer | 🔴 High |
    | Basal Cell Carcinoma | Skin Cancer | 🔴 High |
    | Squamous Cell Carcinoma | Skin Cancer | 🔴 High |
    | Actinic Keratosis | Pre-cancerous | 🟡 Medium |
    | Benign Keratosis | Benign | 🟢 Low |
    | Dermatofibroma | Benign | 🟢 Low |
    | Melanocytic Nevi (Moles) | Benign | 🟢 Low |
    | Vascular Lesion | Vascular | 🟡 Medium |
    """)

    # Connect button to function
    analyze_btn.click(
        fn=analyze_skin,
        inputs=image_input,
        outputs=output_text
    )

# Launch with public link
demo.launch(share=True, debug=False)

/tmp/ipykernel_6699/1502640493.py:13: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Skin Disease Detector", theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1a6f2751dcba124f68.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
